# Semantic Drift — Colab Pipeline

Runs the GPU-dependent parts of the project: InstructPix2Pix baseline edit chains and a SAM segmentation smoke test.

**Before running:**
1. Runtime → Change runtime type → T4 GPU (free tier).
2. Make sure the GitHub repo is public (Settings → General → Danger Zone → Change visibility, on github.com), so the clone cell below doesn't need a token.

In [ ]:
!pip install -q diffusers transformers accelerate segment-anything opencv-python pyyaml

In [ ]:
import os

GITHUB_REPO = "imanshadilshan/semantic_drift"  # update if your GitHub username/repo name differs

clone_dir = "/content/semantic_drift"
if not os.path.exists(clone_dir):
    !git clone https://github.com/{GITHUB_REPO}.git {clone_dir}
else:
    !cd {clone_dir} && git pull

PROJECT_ROOT = f"{clone_dir}/Implementation"

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)

import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU — set Runtime > Change runtime type > GPU")

## Load the dataset

Validates that every image referenced in `edit_instructions.json` actually exists in `raw_images/`.

In [ ]:
from pathlib import Path
from src.data_loader import load_edit_chains

DATA_DIR = Path(PROJECT_ROOT) / "data"
chains = load_edit_chains(str(DATA_DIR / "edit_instructions.json"), str(DATA_DIR / "raw_images"))
print(f"Loaded {len(chains)} chains over {len(set(c['image_id'] for c in chains))} images")

## SAM smoke test

Quick sanity check that segmentation works before Day 14-15's full drift-scoring integration. Downloads the SAM `vit_b` checkpoint (~375MB) the first time it's called.

In [ ]:
from PIL import Image
from src.segment import segment_image

sample_id = chains[0]["image_id"]
sample_image = Image.open(DATA_DIR / "raw_images" / sample_id).convert("RGB")
regions = segment_image(sample_image)
print(f"SAM found {len(regions)} regions in {sample_id}")

## Run baseline edit chains (no mitigation)

Each image is resized to 512x512 (standard for this Stable-Diffusion-based pipeline), then every instruction in its chain is applied in sequence, saving every intermediate step to `results/baseline/`. Safe to re-run after a Colab disconnect — already-completed chains are skipped.

In [ ]:
from src.edit_runner import run_edit_chain

RESULTS_DIR = Path(PROJECT_ROOT) / "results" / "baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for i, chain in enumerate(chains, 1):
    image_id = chain["image_id"]
    chain_type = chain["chain_type"]
    stem = Path(image_id).stem
    out_dir = RESULTS_DIR / f"{stem}_{chain_type}"

    if out_dir.exists() and len(list(out_dir.glob("step*.png"))) == len(chain["instructions"]) + 1:
        continue  # already done

    out_dir.mkdir(parents=True, exist_ok=True)
    image = Image.open(DATA_DIR / "raw_images" / image_id).convert("RGB").resize((512, 512))
    image.save(out_dir / "step0_original.png")

    outputs = run_edit_chain(image, chain["instructions"])
    for step, edited in enumerate(outputs, 1):
        edited.save(out_dir / f"step{step}.png")

    print(f"[{i}/{len(chains)}] {stem} ({chain_type}) done")

print("All baseline chains complete.")

## Compute Drift Scores (Day 14-15)

For each step: SAM segments the pre-edit image once, CLIP identifies which region the instruction targets, and every *other* region is scored for unintended change between the pre- and post-edit crop (same box, both images — a true before/after comparison, not two independent segmentations). Writes `results/baseline_drift_scores.csv` with a per-step score and a cumulative per-chain score.

This is CPU-slow (~180s/chain was measured locally, ~6 hours for all 120) but fast on GPU — that's why it runs here, not on your laptop.

In [ ]:
!cd {PROJECT_ROOT} && python scripts/compute_baseline_drift.py

## Run mitigated edit chains (Day 16-19)

Re-runs the same 120 chains with each mitigation strategy applied at every step:
- **region_locking**: runs the edit on the full image as normal, then reverts every pixel outside the target region back to the pre-edit image (post-hoc correction).
- **masked_conditioning**: crops down to just the target region (plus padding) *before* generation, edits only that crop, then pastes it back — the model never sees the rest of the image (a stricter, generation-time constraint).

Both identify the target region the same way the Drift Score does (SAM + CLIP), so mitigation and measurement agree on what "the target" means. Saved to `results/mitigated/<strategy>/`, same skip-if-done behavior as the baseline run.

In [ ]:
from src import mitigation as mitigation_module

STRATEGIES = {
    "region_locking": mitigation_module.region_locking,
    "masked_conditioning": mitigation_module.masked_conditioning,
}

for strategy_name, step_fn in STRATEGIES.items():
    strategy_dir = Path(PROJECT_ROOT) / "results" / "mitigated" / strategy_name
    strategy_dir.mkdir(parents=True, exist_ok=True)
    print(f"=== {strategy_name} ===")

    for i, chain in enumerate(chains, 1):
        image_id = chain["image_id"]
        chain_type = chain["chain_type"]
        stem = Path(image_id).stem
        out_dir = strategy_dir / f"{stem}_{chain_type}"

        if out_dir.exists() and len(list(out_dir.glob("step*.png"))) == len(chain["instructions"]) + 1:
            continue  # already done

        out_dir.mkdir(parents=True, exist_ok=True)
        image = Image.open(DATA_DIR / "raw_images" / image_id).convert("RGB").resize((512, 512))
        image.save(out_dir / "step0_original.png")

        outputs = run_edit_chain(image, chain["instructions"], step_fn=step_fn)
        for step, edited in enumerate(outputs, 1):
            edited.save(out_dir / f"step{step}.png")

        print(f"[{i}/{len(chains)}] {stem} ({chain_type}) done")

print("All mitigated chains complete.")

## Score the mitigated chains

Same scoring logic as the baseline, run separately against each strategy's saved outputs. Writes `results/region_locking_drift_scores.csv` and `results/masked_conditioning_drift_scores.csv`.

In [ ]:
!cd {PROJECT_ROOT} && python scripts/compute_mitigated_drift.py region_locking
!cd {PROJECT_ROOT} && python scripts/compute_mitigated_drift.py masked_conditioning

## Next steps

Download all three score files back to your laptop before the Colab session ends — the runtime and its disk are wiped on disconnect:

```python
from google.colab import files
for name in ["baseline_drift_scores.csv", "region_locking_drift_scores.csv", "masked_conditioning_drift_scores.csv"]:
    files.download(f"{PROJECT_ROOT}/results/{name}")
```

Day 22-23 (statistical analysis) compares baseline vs. each mitigation strategy from these three files.